# Подготовка данных и разбиение

Нужна папка `train_250k` на Google Диске с `train_manifest.parquet` и 89 парами файлов `*_features.safetensors` и `*_metadata.parquet`. Блокнот проверяет соответствие идентификаторов и координат в манифесте и шардах. Эмбеддинги GeoCLIP (512 признаков) и StreetCLIP (768 признаков) нормируются отдельно и объединяются в 1 280 признаков.

Сначала выделяются 37 500 точек для финального теста, затем остальные 212 500 разбиваются на пять фолдов с учётом страны. Цель представляется единичным направлением ECEF для координат на эллипсоиде WGS84.

В приватном репозитории GitHub выберите **Code → Download ZIP**, загрузите архив в Colab и укажите `ZIP_PATH`, например `/content/train250k-geo-ml-main.zip`. Если репозиторий уже клонирован, укажите `REPO_DIR` или запустите Jupyter из корня проекта. `REPO_URL` можно использовать, если доступ Git уже настроен.

По умолчанию результаты хранятся в `/content/train250k-cv`. Если будете переключать среды выполнения, используйте один и тот же `WORK_DIR` на Google Диске во всех блокнотах, например `/content/drive/MyDrive/train250k-cv`.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = ""  # @param {type:"string"}
ZIP_PATH = ""  # @param {type:"string"}
REPO_DIR = "/content/train250k-geo-ml"  # @param {type:"string"}
WORK_DIR = "/content/train250k-cv"  # @param {type:"string"}
INSTALL_DEPENDENCIES = True  # @param {type:"boolean"}

checkout = Path(REPO_DIR).expanduser()
local_candidates = (checkout, Path.cwd(), Path.cwd().parent)
repo = next((path for path in local_candidates if (path / "src" / "experiment.py").is_file()), None)
if repo is None and ZIP_PATH:
    from tempfile import TemporaryDirectory
    from zipfile import ZipFile

    if checkout.exists():
        raise FileExistsError("Выберите свободный путь REPO_DIR для распаковки.")
    checkout.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=checkout.parent) as tmp:
        with ZipFile(Path(ZIP_PATH).expanduser()) as archive:
            archive.extractall(tmp)
        extracted = next((path for path in Path(tmp).iterdir() if (path / "src" / "experiment.py").is_file()), None)
        if extracted is None:
            raise FileNotFoundError("В архиве GitHub отсутствует src/experiment.py.")
        extracted.rename(checkout)
    repo = checkout
if repo is None:
    if not REPO_URL:
        raise ValueError("Укажите ZIP_PATH, REPO_DIR к готовому репозиторию или REPO_URL с настроенным доступом Git.")
    subprocess.run(["git", "clone", REPO_URL, str(checkout)], check=True)
    repo = checkout

work = Path(WORK_DIR).expanduser().resolve()
env = {**os.environ, "GEO_CV_ROOT": str(work)}
if work.is_relative_to("/content/drive") and not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)


def run_src(script, *arguments):
    subprocess.run([sys.executable, "-u", str(repo / "src" / script), *arguments], cwd=repo, env=env, check=True)

## Папка с исходными данными

Укажите путь `DATA_DIR` из панели файлов Colab после подключения Диска. Если папка доступна по общей ссылке, при необходимости добавьте ярлык на своём Google Диске и используйте путь к ярлыку.

In [ ]:
DATA_DIR = "/content/drive/MyDrive/train_250k"  # @param {type:"string"}
data = Path(DATA_DIR).expanduser()
if data.is_relative_to("/content/drive") and not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
if not (data / "train_manifest.parquet").is_file():
    raise FileNotFoundError(f"Укажите DATA_DIR с train_manifest.parquet. Сейчас: {data}")
env["TRAIN_250K_DIR"] = str(data.resolve())
run_src("prepare.py")

## Разбиение

В следующих блокнотах используйте тот же `WORK_DIR`. Блокноты 01–04 используют только обучающую и валидационную части. Блокнот 05 оценивает фиксированный тест после подбора конфигураций.

In [ ]:
split = json.loads((work / "split.json").read_text())
print(json.dumps(split, ensure_ascii=False, indent=2))